### Part 0: Installing libraries

In [ ]:
%pip install spacy

In [1]:
%pip install spacy-universal-sentence-encoder

  Preparing metadata (setup.py) ... done
  Created wheel for spacy-universal-sentence-encoder: filename=spacy_universal_sentence_encoder-0.4.6-py3-none-any.whl size=16541 sha256=ee12b7db424903d346223efbb0030d98865d37809775b0825e05c1eabe8e81e9
  Stored in directory: /root/.cache/pip/wheels/b8/c5/91/dc7e03329421578e554c8d2635f92fd528531123cce102c7f2
Successfully built spacy-universal-sentence-encoder


In [ ]:
%pip install pyvis

In [ ]:
%pip install geopy

### Part 1: Constructing the similarity index

In order to construct the similarity index, we use spacy, in particular a universal encoder which allows for a more accurate word embedding process. This will help us in detecting true similarity, and getting rid of false-similar sentences, such as sentences that include a large amount of stop-words (e.g sentences that use "the", "a", "he", "it", etc. a lot will not appear as similar, but rather the true meaning of the sentence will be analysed).

In [47]:
import spacy
import numpy as np
import pandas as pd
import networkx as nx
import spacy_universal_sentence_encoder
from tqdm import tqdm
from google.colab import files
from pyvis.network import Network
from IPython.display import display, HTML
import folium
from geopy.geocoders import Nominatim

In [3]:
nlp = spacy_universal_sentence_encoder.load_model('en_use_lg')

Downloaded https://tfhub.dev/google/universal-sentence-encoder-large/5, Total size: 577.10MB



We read the data from the reporting frameworks table that we are working on.

In [57]:
reporting_data = pd.read_excel('ReportingFrameworks.xlsx')

In [53]:
reporting_data

,Framework,Topic,Reference,Recommendation,G_Ref,Scope,Guidance
0,TCFD,Governance,TCFD Governance A,Describe the board’s oversight of climate-rela...,TCFD Governance A1,Generic,Consider including a discussion of the followi...
1,TCFD,Governance,TCFD Governance B,Describe management’s role in assessing and ma...,TCFD Governance B1,Generic,Consider including the following information:\...
2,TCFD,Strategy,TCFD Strategy A,Describe the climate-related risks and opportu...,TCFD Strategy A1,Generic,Organisations should provide the following inf...
3,TCFD,Strategy,TCFD Strategy A,Describe the climate-related risks and opportu...,TCFD Strategy A4,Generic,Organizations should consider providing a desc...
4,TCFD,Strategy,TCFD Strategy A,Describe the climate-related risks and opportu...,TCFD Strategy A5,Banks,Banks should describe significant concentratio...
...,...,...,...,...,...,...,...
550,ESRS,MetricsandTargets,ESRS E4 Metrics and Targets 41,If the undertaking identified material impacts...,AR33: With regard to metrics on the extent and...,NaN,NaN
551,ESRS,MetricsandTargets,ESRS E4 Metrics and Targets 42,The undertaking shall disclose its anticipated...,AR39: The undertaking may include an assessmen...,NaN,NaN
552,ESRS,MetricsandTargets,ESRS E4 Metrics and Targets 43,The information required by paragraph 42 is in...,NaN,NaN,NaN
553,ESRS,MetricsandTargets,ESRS E4 Metrics and Targets 44,The objective of this Disclosure Requirement i...,NaN,NaN,NaN


In [58]:
def similarity_finder(data, columns, run_aggregation = True):
  """
  This is a function that is calculating the similarity index between all of the entries in a specific column, using the spacy framework.

  Parameters:
  "data" represents the data that we are putting in the format provided to us by Lloyd.
  "columns" contains, in order:
  - the column which contains the framework
  - the column which contains the topic
  - the column which contains the exact reference
  - the column which contains the entry that we want to compare (which will initially be the recommendation, but may become the guidance)

  Outputs:
  The output will be similar to the similarrequrements.csv file. This output will then be fed into a download function, in case we want to download the file,
  and a networkx code which will create a network of all of these links.
  """
  framework_column = columns[0]
  topic_column = columns[1]
  reference_column = columns[2]
  comparison_column = columns[3]

  if(run_aggregation):
    data = data.replace("ESRS E1", "ESRS")
    data = data.replace("ESRS E4", "ESRS")
    data = data.replace("IFRS S1", "IFRS")
    data = data.replace("IFRS S2", "IFRS")

  n = int(len(data)*(len(data)-1)/2)
  similarity_list = []

  # Create a progress bar for the total number of comparisons, while also creating the similarity table file
  with tqdm(total=n, desc="Comparing pairs") as pbar:
    for i in range(len(data)):
      curr_framework = data.iloc[i][framework_column]
      curr_topic = data.iloc[i][topic_column]
      curr_reference = data.iloc[i][reference_column]
      curr_comparison = data.iloc[i][comparison_column]
      curr_nlp = nlp(curr_comparison)
      for j in range(i+1, len(data)):
        next_framework = data.iloc[j][framework_column]
        next_topic = data.iloc[j][topic_column]
        next_reference = data.iloc[j][reference_column]
        next_comparison = data.iloc[j][comparison_column]
        next_nlp = nlp(next_comparison)
        if (next_framework != curr_framework) and (next_topic == curr_topic): #We are getting rid of the similarities within the same framework, which are bound to be high, and we only calculate similarities where they are useful, in the same topic
          similarity_score = curr_nlp.similarity(next_nlp)
          similarity_list.append({"Topic":curr_topic, "Framework 1": curr_framework, "Reference 1": curr_reference, "Recommendation 1": curr_comparison, "Framework 2": next_framework, "Reference 2": next_reference, "Recommendation 2": next_comparison, "Similarity": similarity_score})
        pbar.update(1)  # Update the progress bar after each comparison

  similarity_table = pd.DataFrame(similarity_list)
  similarity_table.drop_duplicates(inplace=True)
  similarity_table.reset_index(drop=True, inplace=True)
  return similarity_table

In [59]:
similarity_table = similarity_finder(reporting_data, ["Framework", "Topic", "Reference", "Recommendation"])

Comparing pairs: 100%|██████████| 153735/153735 [01:15<00:00, 2028.48it/s]


In [60]:
def download_table(table, filename):
  """
  Downloads tables as a CSV file in Google Colab.

  Parameters:
  - table: pandas DataFrame to download
  - filename: name of the file to save (without extension)

  """
  # Save the DataFrame to a CSV file
  table.to_csv(filename, index=False)

  # Download the file
  files.download(filename)

  print(f"File '{filename}' has been downloaded!")

In [61]:
download_table(similarity_table, "similarity_table.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'similarity_table.csv' has been downloaded!


### Part 2: Constructing the network of similarity

Below, what we will be creating is a similarity metric between different frameworks. The list of topics from which one can choose below is as follows: Disclosure, Governance, Metrics and Targets, Risk Management and Strategy.

In [62]:
def similarity_network_table(original_list = ["Disclosure", "Governance", "Metrics and Targets", "Risk Management", "Strategy"], original_data=similarity_table):
  """
  This is a function that is calculating the data required to build a network of the similarities of different frameworks, based on a list of topics.

  Parameters:
  "list" is the list of topics on which we want to base the similarity. It can contain all topics, but it must include at least one. It's originally defined as the list of all topics.
  "data" is the underlying data, which in this file should be the similarity table found before. It's originally defined as the similarity_table. We are assuming that the data is in the format of the similarity table.

  Outputs:
  A table containing the similarity between all of the frameworks, based on the list of topics. This similarity is calculated as the average of the similarities of all recommendations
  in the chosen topics.
  """
  filtered_data = original_data
  for i in range(len(filtered_data)):
    if not(filtered_data["Topic"][i] in original_list):
      filtered_data.drop(i, inplace=True)
  filtered_data.reset_index(drop=True, inplace=True)
  framework_1 = pd.unique(filtered_data["Framework 1"])
  framework_2 = pd.unique(filtered_data["Framework 2"])
  all_frameworks = np.concatenate((framework_1, framework_2))
  all_frameworks = pd.unique(all_frameworks)
  similarity_list = []
  with tqdm(total=len(all_frameworks)*(len(all_frameworks)-1)/2, desc="Comparing pairs") as pbar:
    for i in range(len(all_frameworks)):
      curr_framework = all_frameworks[i]
      for j in range(i+1,len(all_frameworks)):
        next_framework = all_frameworks[j]
        similarity = filtered_data.loc[(filtered_data["Framework 1"] == curr_framework) & (filtered_data["Framework 2"] == next_framework)]["Similarity"].mean()
        similarity_list.append({"Framework 1": curr_framework, "Framework 2": next_framework, "Similarity": similarity})
        pbar.update(1)

  similarity_network = pd.DataFrame(similarity_list)
  similarity_network.fillna(0, inplace=True)
  similarity_network.drop_duplicates(inplace=True)
  similarity_network.reset_index(drop=True, inplace=True)
  return similarity_network

In [63]:
similarity_network_table()

Comparing pairs: 100%|██████████| 28/28.0 [00:00<00:00, 1034.62it/s]


,Framework 1,Framework 2,Similarity
0,TCFD,TNFD,0.541245
1,TCFD,PRA,0.235241
2,TCFD,IFRS,0.249970
3,TCFD,TPT,0.137106
4,TCFD,BMA,0.195141
5,TCFD,MAS,0.278410
6,TCFD,ESRS,0.223529
7,TNFD,PRA,0.229915
8,TNFD,IFRS,0.235605
9,TNFD,TPT,0.175116


We are now going to asign to each framework a country where it either originated from or is in use.

In [71]:
country_dictionary = {
    "Framework": ["TCFD", "PRA", "BMA", "MAS", "ESRS"],
    "Country": ["USA", "United Kingdom", "Bermuda", "Singapore", "Germany"]
}

In [72]:
country_df = pd.DataFrame(country_dictionary)
merged_df = pd.merge(similarity_network_table(), country_df, left_on='Framework 1', right_on='Framework', how='left')
merged_df.rename(columns={'Country': 'Country 1'}, inplace=True)
merged_df.drop(columns=['Framework'], inplace=True)
merged_df = pd.merge(merged_df, country_df, left_on='Framework 2', right_on='Framework', how='left')
merged_df.rename(columns={'Country': 'Country 2'}, inplace=True)
merged_df.drop(columns=['Framework'], inplace=True)
merged_df.dropna(inplace=True)
merged_df.reset_index(drop=True, inplace=True)
display(merged_df.head())

Comparing pairs: 100%|██████████| 28/28.0 [00:00<00:00, 330.41it/s]


,Framework 1,Framework 2,Similarity,Country 1,Country 2
0,TCFD,PRA,0.235241,USA,United Kingdom
1,TCFD,BMA,0.195141,USA,Bermuda
2,TCFD,MAS,0.278410,USA,Singapore
3,TCFD,ESRS,0.223529,USA,Germany
4,PRA,BMA,0.000000,United Kingdom,Bermuda


Our goal now is to build an interactive map of the different similarities, over the map of the world.

In [73]:
# Identify unique countries
unique_countries = pd.concat([merged_df['Country 1'], merged_df['Country 2']]).dropna().unique()

# Initialize Nominatim geocoder
geolocator = Nominatim(user_agent="framework_network_app")

# Get coordinates for each country
country_coordinates = {}
for country in unique_countries:
    try:
        location = geolocator.geocode(country)
        if location:
            country_coordinates[country] = {'latitude': location.latitude, 'longitude': location.longitude}
        else:
            print(f"Could not find coordinates for {country}")
            country_coordinates[country] = {'latitude': None, 'longitude': None}
    except Exception as e:
        print(f"Error getting coordinates for {country}: {e}")
        country_coordinates[country] = {'latitude': None, 'longitude': None}

# Display the collected coordinates
display(country_coordinates)

{'USA': {'latitude': 39.7837304, 'longitude': -100.445882},
 'United Kingdom': {'latitude': 54.7023545, 'longitude': -3.2765753},
 'Bermuda': {'latitude': 32.3040273, 'longitude': -64.7563086},
 'Singapore': {'latitude': 1.357107, 'longitude': 103.8194992},
 'Germany': {'latitude': 51.1638175, 'longitude': 10.4478313}}

In [74]:
G = nx.Graph()

for index, row in merged_df.iterrows():
    framework1 = row['Framework 1']
    framework2 = row['Framework 2']
    similarity = row['Similarity']
    country1 = row['Country 1']
    country2 = row['Country 2']

    # Add nodes with country attribute
    if framework1 not in G:
        G.add_node(framework1, country=country1)
    if framework2 not in G:
        G.add_node(framework2, country=country2)

    # Add edge with similarity attribute
    G.add_edge(framework1, framework2, similarity=similarity)

print("NetworkX graph created with nodes and edges.")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")

NetworkX graph created with nodes and edges.
Number of nodes: 5
Number of edges: 10


In [75]:
# Create a base map centered at a reasonable initial location
m = folium.Map(location=[0, 0], zoom_start=2)

# Add markers for each framework based on country coordinates
for node, data in G.nodes(data=True):
    country = data.get('country')
    if country and country in country_coordinates and country_coordinates[country]['latitude'] is not None and country_coordinates[country]['longitude'] is not None:
        folium.Marker(
            location=[country_coordinates[country]['latitude'], country_coordinates[country]['longitude']],
            popup=node,
            tooltip=node
        ).add_to(m)
    else:
        print(f"Could not add marker for {node} due to missing or invalid country coordinates.")

# Add edges as lines on the map, with thickness based on similarity
for u, v, data in G.edges(data=True):
    similarity = data.get('similarity', 0)
    country_u = G.nodes[u].get('country')
    country_v = G.nodes[v].get('country')

    if (country_u and country_u in country_coordinates and country_coordinates[country_u]['latitude'] is not None and country_coordinates[country_u]['longitude'] is not None and
        country_v and country_v in country_coordinates and country_coordinates[country_v]['latitude'] is not None and country_coordinates[country_v]['longitude'] is not None):

        loc_u = [country_coordinates[country_u]['latitude'], country_coordinates[country_u]['longitude']]
        loc_v = [country_coordinates[country_v]['latitude'], country_coordinates[country_v]['longitude']]

        # Scale line thickness based on similarity
        line_weight = similarity * 5 # Adjust scaling factor as needed

        folium.PolyLine([loc_u, loc_v], weight=line_weight, opacity=0.7, tooltip=f"{u} - {v}: {similarity:.2f}").add_to(m)
    else:
        print(f"Could not draw edge between {u} and {v} due to missing or invalid country coordinates.")

# Save the map as an HTML file
map_file = "framework_similarity_map.html"
m.save(map_file)
print(f"Map saved to {map_file}")

Map saved to framework_similarity_map.html


In [76]:
map_file = "framework_similarity_map.html"
files.download(map_file)
display(HTML(filename=map_file))